# 04 — Auto-Tuning Pallas Kernels

This notebook demonstrates the `pallas-forge` auto-tuning framework:

1. Define a configuration search space
2. Run systematic benchmarks across all configs
3. Generate performance heatmaps
4. Analyze results to understand *why* some configs are faster

> **Note**: On CPU (interpret mode), timing differences are minimal because there's no real
> MXU/VMEM hardware. On TPU, you'd see 3-5x swings between configs. The framework and
> visualization pipeline work identically regardless.

In [ ]:
# ============================================================
# Colab / TPU Setup — run this cell first!
# ============================================================
# On Colab: Runtime > Change runtime type > TPU
#
# If running locally with pallas-forge already installed, skip this cell.
# ============================================================

import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "jax[tpu]", "-f",
        "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/<YOUR_USERNAME>/pallas-forge.git#egg=pallas-forge[viz]",
    ])
    print("Colab setup complete!")
else:
    print("Running locally — skipping Colab install.")

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt

from pallas_forge import tiled_matmul, fused_swiglu
from pallas_forge.tune import tune, TuneConfig
from pallas_forge.tune.runner import BenchmarkRunner
from pallas_forge.tune.report import TuneReport

## 1. Define the search space

`TuneConfig` defines which parameter combinations to explore.

In [ ]:
# From a Python dict
config = TuneConfig.from_dict({
    "block_m": [32, 64, 128],
    "block_k": [32, 64, 128],
    "block_n": [32, 64, 128],
})

print(f"Parameters: {config.param_names}")
print(f"Total combinations: {config.total_combinations}")

# Add constraints to filter invalid combos
config.add_constraint(lambda p: p["block_m"] >= 32)
config.add_constraint(lambda p: p["block_k"] <= p["block_m"])  # K shouldn't exceed M

valid_configs = config.grid()
print(f"Valid combinations (after constraints): {len(valid_configs)}")
print(f"\nFirst 5 configs:")
for c in valid_configs[:5]:
    print(f"  {c}")

## 2. Run auto-tuning with `tune()`

The `tune()` function is the single entry point. It:
1. Generates configs from the search space
2. Benchmarks each one (with warmup + statistical timing)
3. Returns a `TuneReport` for analysis

In [ ]:
# Problem setup
M, K, N = 128, 128, 128  # Small for CPU demo

def input_fn(cfg):
    """Create inputs for each config."""
    key = jax.random.PRNGKey(0)
    k1, k2 = jax.random.split(key)
    x = jax.random.normal(k1, (M, K), dtype=jnp.float32)
    w = jax.random.normal(k2, (K, N), dtype=jnp.float32)
    return (x, w)

def kernel_fn(x, w, *, block_m, block_k, block_n):
    """Kernel wrapper matching tune() calling convention."""
    return tiled_matmul(x, w, block_m=block_m, block_k=block_k, block_n=block_n)

# Run auto-tuning (reduced iterations for CPU)
report = tune(
    kernel_fn=kernel_fn,
    input_fn=input_fn,
    config=config,
    strategy="grid",
    n_warmup=2,
    n_repeat=5,
)

## 3. Analyze results

In [ ]:
# Best and worst configs
best = report.best(3)
worst = report.worst(3)

print("Top 3 fastest configs:")
for i, r in enumerate(best):
    print(f"  {i+1}. {r.config} -> {r.median_ms:.3f} ms")

print(f"\nWorst 3 configs:")
for i, r in enumerate(worst):
    print(f"  {i+1}. {r.config} -> {r.median_ms:.3f} ms")

print(f"\nSpeedup range: {report.speedup_range:.2f}x")
print(f"(On TPU, this would typically be 3-5x)")

## 4. Generate performance heatmaps

Heatmaps are the visual centerpiece — they show how parameter choices affect performance.

In [ ]:
# Heatmap: block_m vs block_n
fig = report.heatmap(
    "block_m", "block_n",
    metric="median_ms",
    title=f"MatMul Latency (ms) — {M}x{K}x{N}",
    cmap="YlOrRd_r",
)
plt.show()
print("Yellow = fast, Red = slow")

In [ ]:
# Heatmap: block_m vs block_k
fig = report.heatmap(
    "block_m", "block_k",
    metric="median_ms",
    title=f"MatMul Latency by block_m vs block_k",
    cmap="YlOrRd_r",
)
plt.show()

## 5. Export results

In [ ]:
import tempfile, json
from pathlib import Path

tmpdir = tempfile.mkdtemp()

# JSON export
json_path = Path(tmpdir) / "matmul_results.json"
report.to_json(json_path)

# CSV export
csv_path = Path(tmpdir) / "matmul_results.csv"
report.to_csv(csv_path)

# Heatmap to file
heatmap_path = Path(tmpdir) / "heatmap.png"
report.heatmap("block_m", "block_n", save_path=heatmap_path)

# Show JSON structure
with open(json_path) as f:
    data = json.load(f)
print(f"Exported {len(data)} results to JSON")
print(f"\nSample entry:")
print(json.dumps(data[0], indent=2))

print(f"\nFiles saved to: {tmpdir}")

## 6. Random search for large spaces

When the search space is too large for exhaustive grid search, use random sampling.

In [ ]:
from pallas_forge.tune.search import RandomSearch

large_config = TuneConfig.from_dict({
    "block_m": [16, 32, 64, 128, 256],
    "block_k": [16, 32, 64, 128, 256],
    "block_n": [16, 32, 64, 128, 256],
})

print(f"Total space: {large_config.total_combinations} configs")

# Sample 10 random configs
random_strategy = RandomSearch(n_trials=10, seed=42)
sampled = random_strategy.generate(large_config)
print(f"Sampled: {len(sampled)} configs")

for c in sampled:
    print(f"  {c}")

## 7. Custom kernel tuning — SwiGLU example

In [ ]:
# Tune SwiGLU
swiglu_config = TuneConfig.from_dict({
    "block_m": [16, 32, 64],
    "block_n": [32, 64, 128],
})

BS, D, FFN = 32, 64, 128

def swiglu_input_fn(cfg):
    key = jax.random.PRNGKey(0)
    k1, k2, k3 = jax.random.split(key, 3)
    return (
        jax.random.normal(k1, (BS, D), dtype=jnp.float32),
        jax.random.normal(k2, (D, FFN), dtype=jnp.float32),
        jax.random.normal(k3, (D, FFN), dtype=jnp.float32),
    )

def swiglu_kernel_fn(x, w_gate, w_up, *, block_m, block_n):
    return fused_swiglu(x, w_gate, w_up, block_m=block_m, block_n=block_n)

swiglu_report = tune(
    kernel_fn=swiglu_kernel_fn,
    input_fn=swiglu_input_fn,
    config=swiglu_config,
    n_warmup=2,
    n_repeat=5,
)

fig = swiglu_report.heatmap(
    "block_m", "block_n",
    metric="median_ms",
    title=f"SwiGLU Latency (ms) — {BS}x{D}x{FFN}",
)
plt.show()

## 8. YAML config files

For reproducible experiments, define configs in YAML and version-control them.

In [ ]:
import tempfile

yaml_content = """
block_m: [32, 64, 128]
block_n: [32, 64, 128]
block_k: [32, 64]
"""

# Write to temp file
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(yaml_content)
    yaml_path = f.name

# Load from YAML
yaml_config = TuneConfig.from_yaml(yaml_path)
print(f"Loaded from YAML: {yaml_config.param_names}")
print(f"Total combinations: {yaml_config.total_combinations}")

## Key takeaways

1. **`tune()`** is the single entry point — pass any kernel + input_fn + config space
2. **Heatmaps** reveal which parameters matter most (on TPU, block_m and block_n typically dominate)
3. **Grid search** for small spaces, **random search** for large ones
4. **YAML configs** enable reproducible experiments
5. On CPU, timing differences are small; on TPU, expect 3-5x swings between best and worst configs

### Next steps (on TPU)

- Run with larger matrices (2048x2048+) on TPU to see real performance differences
- Add `flops_fn` and `bytes_fn` to compute TFLOPS and bandwidth
- Generate roofline charts to classify each config as compute-bound or memory-bound
- Capture XProf traces for top-N configs to understand hardware utilization